# 01 · Explorer les trois sources

Un assureur fictif à Madagascar gère des fiches dans Auto (PostgreSQL), Life
(MySQL) et Autre (CSV). Relevez les colonnes, les types et les formats.

Exécutez les cellules dans l’ordre. Le mode rapide affiche des schémas et des
échantillons. Pour compter toutes les lignes et examiner les valeurs manquantes,
remplacez la ligne de configuration par « PROFIL_COMPLET = True », puis
réexécutez toutes les cellules.

In [1]:
import csv
import os
import re
from collections import Counter
from contextlib import closing, contextmanager
from getpass import getpass
from html import escape
from itertools import islice
from pathlib import Path
from time import perf_counter

import mysql.connector
import psycopg
from IPython.display import HTML, display

PROFIL_COMPLET = os.getenv("TP_PROFIL_COMPLET") == "1"
TAILLE_ECHANTILLON = 4
LIMITE_APERCU_CSV = 1_000
CHEMIN_CSV = Path("data/clients_autre.csv")
if not CHEMIN_CSV.is_file():
    CHEMIN_CSV = Path("data/clients_autre_leger.csv")
if not CHEMIN_CSV.is_file():
    CHEMIN_CSV = Path("..") / "data" / "clients_autre.csv"
if not CHEMIN_CSV.is_file():
    CHEMIN_CSV = Path("..") / "data" / "clients_autre_leger.csv"
if not CHEMIN_CSV.is_file():
    raise FileNotFoundError("Placer data/clients_autre.csv (ou data/clients_autre_leger.csv) à la racine du projet.")

print("Mode complet" if PROFIL_COMPLET else "Mode rapide")

Mode rapide


## Se connecter en lecture seule

Utilisez les comptes de lecture fournis pour le TP et les variables du README.
Si le mot de passe est absent de l’environnement, il est demandé de manière masquée.
Les curseurs et connexions sont fermés à la fin de chaque lecture.

In [2]:
def acces(prefixe, port_defaut):
    utilisateur = os.getenv(f"{prefixe}_USER")
    if not utilisateur:
        raise RuntimeError(f"Définir {prefixe}_USER avant de lancer JupyterLab.")
    mot_de_passe = os.getenv(f"{prefixe}_PASSWORD")
    if mot_de_passe is None:
        mot_de_passe = getpass(f"Mot de passe pour {prefixe} : ")
    return {
        "host": os.getenv(f"{prefixe}_HOST", "127.0.0.1"),
        "port": int(os.getenv(f"{prefixe}_PORT", str(port_defaut))),
        "user": utilisateur,
        "password": mot_de_passe,
    }


@contextmanager
def ouvrir_auto():
    try:
        connexion = psycopg.connect(
            **acces("TP_AUTO", 5432), dbname="tp_assurance_auto", connect_timeout=5,
        )
    except psycopg.Error:
        raise RuntimeError("Connexion Auto impossible. Vérifier les accès et le serveur.") from None
    with closing(connexion):
        connexion.read_only = True
        try:
            yield connexion
        finally:
            connexion.rollback()


@contextmanager
def ouvrir_life():
    try:
        connexion = mysql.connector.connect(
            **acces("TP_LIFE", 3306), database="tp_assurance_life",
            connection_timeout=5, charset="utf8mb4",
        )
    except mysql.connector.Error:
        raise RuntimeError("Connexion Life impossible. Vérifier les accès et le serveur.") from None
    with closing(connexion):
        connexion.start_transaction(readonly=True)
        try:
            yield connexion
        finally:
            connexion.rollback()

## Explorer Auto et Life

Le catalogue donne les colonnes, les types et la possibilité d’une valeur
NULL. Les aperçus sont bornés par LIMIT. En mode complet, les agrégats sont
calculés côté serveur. Pour le texte, les chaînes vides après retrait des
espaces sont comptées avec les NULL.

In [3]:
def tableau(colonnes, lignes):
    def texte(valeur):
        return "NULL" if valeur is None else escape(str(valeur))

    entete = "".join(f"<th>{texte(colonne)}</th>" for colonne in colonnes)
    corps = "".join(
        "<tr>" + "".join(f"<td>{texte(valeur)}</td>" for valeur in ligne) + "</tr>"
        for ligne in lignes
    )
    display(HTML(f"<table><thead><tr>{entete}</tr></thead><tbody>{corps}</tbody></table>"))


def explorer_base(ouvrir, nom, schema, table, reference, apercu, guillemet):
    with ouvrir() as connexion, connexion.cursor() as curseur:
        curseur.execute(
            "SELECT column_name, data_type, is_nullable FROM information_schema.columns "
            "WHERE table_schema = %s AND table_name = %s ORDER BY ordinal_position",
            (schema, table),
        )
        colonnes = curseur.fetchall()
        if not colonnes:
            raise RuntimeError(f"{nom} : table introuvable ou accès en lecture insuffisant.")

        print(f"{nom} · {schema}.{table}")
        tableau(("Colonne", "Type SQL", "NULL autorisé"), colonnes)
        curseur.execute(
            f"SELECT {', '.join(apercu)} FROM {schema}.{table} ORDER BY {reference} LIMIT %s",
            (TAILLE_ECHANTILLON,),
        )
        tableau(apercu, curseur.fetchall())
        if not PROFIL_COMPLET:
            return None

        expressions = []
        for colonne, type_sql, _ in colonnes:
            identifiant = guillemet + colonne.replace(guillemet, guillemet * 2) + guillemet
            valeur = identifiant
            if type_sql in {"text", "character varying", "character", "varchar", "char"}:
                valeur = f"NULLIF(TRIM({identifiant}), '')"
            expressions.append(f"COUNT(*) - COUNT({valeur})")
        curseur.execute(f"SELECT COUNT(*), {', '.join(expressions)} FROM {schema}.{table}")
        volume, *absences = curseur.fetchall()[0]
        print(f"{nom} · {volume:,} fiches".replace(",", " "))
        tableau(("Colonne", "Valeurs absentes"), zip((c[0] for c in colonnes), absences))
        return {"volume": volume, "absences": dict(zip((c[0] for c in colonnes), absences))}

In [4]:
profil_auto = explorer_base(
    ouvrir_auto, "Auto", "tp_assurance", "clients_auto", "id_client_auto",
    ("id_client_auto", "numero_cin", "ne_le", "mobile", "sexe", "formule_auto"), '"',
)

Auto · tp_assurance.clients_auto


Colonne,Type SQL,NULL autorisé
id_client_auto,bigint,NO
numero_cin,text,YES
nom_assure,text,NO
prenoms_assure,text,NO
ne_le,date,NO
ne_a,text,YES
ville_residence,text,NO
mobile,text,YES
courriel,text,YES
immatriculation,text,NO


id_client_auto,numero_cin,ne_le,mobile,sexe,formule_auto
1000001,CIN- 800003046307,1982-08-01,00 54 666 59,Féminin,Tiers étendu
1000002,CIN- 800003051636,1988-12-23,00 54 762 22,Masculin,Tous risques
1000003,CIN- 800003056965,1995-05-17,00 54 857 85,Masculin,Tiers
1000004,CIN- 800003062294,2001-10-08,00 54 953 48,Féminin,Tiers étendu


In [5]:
profil_life = explorer_base(
    ouvrir_life, "Life", "tp_assurance_life", "clients_life", "ref_adherent",
    ("ref_adherent", "cin_numerique", "date_naiss", "tel_contact", "genre", "produit"), chr(96),
)

Life · tp_assurance_life.clients_life


Colonne,Type SQL,NULL autorisé
ref_adherent,bigint,NO
cin_numerique,bigint,YES
nom_adherent,text,NO
prenoms_adherent,text,NO
date_naiss,int,NO
lieu_naiss,text,YES
commune_domicile,text,NO
tel_contact,text,YES
email_contact,text,YES
produit,text,NO


ref_adherent,cin_numerique,date_naiss,tel_contact,genre,produit
3000001,800001984449,19640312,+000003561133,1,Épargne vie
3000002,800001994450,19760313,+000003579080,0,Retraite individuelle
3000003,800002004451,19880314,+000003597027,0,Épargne vie
3000004,800002014452,20000315,+000003614974,0,Retraite individuelle


## Explorer Autre en flux

Le lecteur CSV renvoie du texte. L’aperçu lit au plus 1 000 fiches.
En mode complet, une seule lecture suffit pour compter les lignes, les
absences et les formes observées. Quatre exemples au plus sont conservés
par secteur. Les champs vides non applicables à un secteur sont comptés
à part. Les motifs vérifient la présentation des valeurs.

In [9]:
FORMATS = {
    "piece_identite": re.compile(r"[0-9]{12}"),
    "date_naissance": re.compile(r"[0-9]{2}-[0-9]{2}-[0-9]{4}"),
    "contact_tel": re.compile(r"[0-9]{2}-[0-9]{2}-[0-9]{3}-[0-9]{2}"),
    "contact_mail": re.compile(r"[^@\s]+@[^@\s]+\.[^@\s]+"),
    "code_sexe": re.compile(r"[MF]"),
    "superficie_ha": re.compile(r"[0-9]+,[0-9]{2}"),
}
CHAMPS_SECTEUR = {
    "AGRICOLE": {"culture", "superficie_ha", "garantie_agricole"},
    "MARITIME": {"type_navire", "port_attache", "garantie_maritime"},
}
APERCU_CSV = (
    "ref_fiche", "plateforme", "piece_identite", "date_naissance",
    "contact_tel", "code_sexe", "culture", "type_navire",
)


def explorer_csv(chemin, complet):
    debut = perf_counter()
    secteurs, absences, non_applicables = Counter(), Counter(), Counter()
    formats = {champ: Counter() for champ in FORMATS}
    exemples = {secteur: [] for secteur in CHAMPS_SECTEUR}
    volume = 0
    with chemin.open(encoding="utf-8", newline="") as fichier:
        lecteur = csv.reader(fichier, delimiter=";", strict=True)
        entete = next(lecteur, [])
        requis = set(APERCU_CSV).union(FORMATS, *CHAMPS_SECTEUR.values())
        if len(entete) != len(set(entete)) or not requis <= set(entete):
            raise ValueError("En-tête du CSV inattendu.")
        lignes = lecteur if complet else islice(lecteur, LIMITE_APERCU_CSV)
        for ligne in lignes:
            if len(ligne) != len(entete):
                raise ValueError(f"Nombre de champs incorrect à la ligne {lecteur.line_num}.")
            fiche = dict(zip(entete, ligne))
            secteur = fiche["plateforme"]
            if secteur not in CHAMPS_SECTEUR:
                raise ValueError(f"Secteur inattendu à la ligne {lecteur.line_num}.")
            volume += 1
            secteurs[secteur] += 1
            if len(exemples[secteur]) < TAILLE_ECHANTILLON:
                exemples[secteur].append(tuple(fiche[champ] for champ in APERCU_CSV))
            if complet:
                autre_secteur = "MARITIME" if secteur == "AGRICOLE" else "AGRICOLE"
                for champ, valeur in fiche.items():
                    if not valeur.strip():
                        compteur = non_applicables if champ in CHAMPS_SECTEUR[autre_secteur] else absences
                        compteur[champ] += 1
                for champ, motif in FORMATS.items():
                    valeur = fiche[champ]
                    if valeur.strip():
                        formats[champ]["conforme" if motif.fullmatch(valeur) else "autre"] += 1

    return {
        "entete": entete, "volume": volume, "secteurs": secteurs, "exemples": exemples,
        "absences": absences, "non_applicables": non_applicables, "formats": formats,
        "secondes": perf_counter() - debut,
    }


profil_autre = explorer_csv(CHEMIN_CSV, PROFIL_COMPLET)
volume = f"{profil_autre['volume']:,}".replace(",", " ")
print(f"Autre · {volume} fiches lues en {profil_autre['secondes']:.2f} s")
tableau(("Colonne", "Type lu par csv"), ((c, "str") for c in profil_autre["entete"]))
tableau(("Secteur", "Fiches lues"), sorted(profil_autre["secteurs"].items()))
for secteur, exemples in profil_autre["exemples"].items():
    print(secteur)
    tableau(APERCU_CSV, exemples)

if PROFIL_COMPLET:
    tableau(("Colonne", "Absences", "Vides non applicables"), (
        (c, profil_autre["absences"][c], profil_autre["non_applicables"][c])
        for c in profil_autre["entete"]
    ))
    tableau(("Contrôle de forme", "Conformes", "Autres formes"), (
        (c, stats["conforme"], stats["autre"]) for c, stats in profil_autre["formats"].items()
    ))

Autre · 1 000 fiches lues en 0.01 s


Colonne,Type lu par csv
ref_fiche,str
plateforme,str
piece_identite,str
patronyme,str
autres_noms,str
date_naissance,str
commune_origine,str
localite,str
contact_tel,str
contact_mail,str


Secteur,Fiches lues
AGRICOLE,500
MARITIME,500


AGRICOLE


ref_fiche,plateforme,piece_identite,date_naissance,contact_tel,code_sexe,culture,type_navire
EXT-5000001,AGRICOLE,800005316023,19-04-1970,00-95-397-11,M,Riz,
EXT-5000003,AGRICOLE,800005344201,12-02-2004,00-95-902-77,M,Maïs,
EXT-5000005,AGRICOLE,800005372379,28-01-1993,00-96-408-43,F,Manioc,
EXT-5000007,AGRICOLE,800005400557,13-01-1982,00-96-914-09,M,Vanille,


MARITIME


ref_fiche,plateforme,piece_identite,date_naissance,contact_tel,code_sexe,culture,type_navire
EXT-5000002,MARITIME,800005330112,17-03-1987,00-95-649-94,F,,Vedette de pêche
EXT-5000004,MARITIME,800005358290,02-03-1976,00-96-155-60,M,,Caboteur
EXT-5000006,MARITIME,800005386468,15-02-1965,00-96-661-26,F,,Pirogue
EXT-5000008,MARITIME,800005414646,11-12-1998,00-97-166-92,F,,Vedette de pêche


## Comparer vos observations

| Attribut | Auto | Life | Autre |
| --- | --- | --- | --- |
| Pièce d’identité | Texte avec préfixe | Entier | Texte de chiffres |
| Naissance | Date SQL | Entier AAAAMMJJ | Texte JJ-MM-AAAA |
| Téléphone | Groupes avec espaces | Préfixe fictif +000 | Groupes avec tirets |
| Sexe | Masculin / Féminin | 0 / 1 | M / F |

Relevez une différence de type et un exemple de valeur absente ou non
applicable. Expliquez pourquoi les références des fiches sont propres
à leur source.

In [10]:
if PROFIL_COMPLET:
    observations = (
        ("Auto", profil_auto["volume"], 100_000),
        ("Life", profil_life["volume"], 100_000),
        ("Autre", profil_autre["volume"], 200_000),
    )
    tableau(("Source", "Fiches observées", "Fiches attendues", "Écart"),
            ((s, v, a, v - a) for s, v, a in observations))
